# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark

In [ ]:
spark = (
    SparkSession
        .builder
        .appName("SessaoLocal")
        .master("local[12]")
        .config("spark.driver.memory", "24g")
        .getOrCreate()
)
spark

## Arquivos de dados

### Dados de origem

In [ ]:
folder_oi = Path("/assets/monitoring/cdr/cdr_bruto/Semana101/Oi/CDR")
_files_oi = [str(file) for file in folder_oi.glob("*.gz")]
df = pd.DataFrame(_files_oi, columns=["file_path"])
df["exchange_id"] = df["file_path"].str.extract(r"/([A-Z]{4}).")
files_oi = df.groupby("exchange_id").sample(1,random_state=42)["file_path"].to_list()
files_oi

### Arquivos intermediário e final

In [ ]:
extracted_oi = "/assets/data/cdr/teletools/db/extraidos/tropico_oi_extraido.parquet"
transformed_oi = "/assets/data/cdr/teletools/db/transformados/tropico_oi_transformado.parquet"

# Análise

## Dados brutos

In [ ]:
rdd = spark.sparkContext.textFile(",".join(files_oi))
df_raw = rdd.map(lambda x: (x,)).toDF(["value"])
df_raw.show(5, truncate=False)

## Extração

In [ ]:
extractor = CDRTextExtractor(spark)
extractor.extract(files_oi, extracted_oi, "stfc_tropico_oi")

In [ ]:
df_extracted = spark.read.parquet(extracted_oi)
df_extracted.show(5)

## Transformação

In [ ]:
transformer = CDRTransformer(spark)
transformer.transform_stfc_tropico_oi(extracted_oi, transformed_oi)

In [ ]:
df_transformed = spark.read.parquet(transformed_oi)
df_transformed.limit(5).toPandas().dropna(how="all",axis=1)